# Chunking Strategies for RAG
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/53-chunking-strategies/chunking_workbook.ipynb)

**Chunking** is the single most impactful architectural decision in a RAG pipeline. How you slice a document determines what the retriever can find — and therefore what the LLM can answer. By the end of this workshop you will understand *why* chunking matters, *how* four major strategies differ, and *how* to measure their impact on real retrieval quality.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why chunking exists and what makes a good chunk |
| 2 | **Fixed-Size** — `CharacterTextSplitter`: fast but boundary-blind |
| 3 | **Recursive** — `RecursiveCharacterTextSplitter`: structure-aware default |
| 4 | **Sentence-Window** — embed sentences, store neighbors |
| 5 | **Semantic** — cosine-boundary detection for topic shifts |
| 6 | **Side-by-Side Comparison** — same document, four strategies |
| 7 | **RAG Quality Check** — index all four, compare answers |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with requirements already installed), **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401  
> Gao, Y., Xiong, Y., et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey.* https://arxiv.org/abs/2312.10997  
> LangChain Text Splitters: https://python.langchain.com/docs/concepts/text_splitters/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-text-splitters",
            "chromadb",
            "numpy",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os
import re
from typing import List

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running embedding or LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why Chunking Exists

---

### The Context Budget Problem

LLMs have a fixed context window (e.g., 128k tokens for GPT-4o). Even when that window is large:

1. **Cost** — sending a 500-page document when only two paragraphs are relevant wastes tokens and money
2. **Quality** — LLMs perform worse when buried in irrelevant text (*lost-in-the-middle* effect¹)
3. **Retrieval precision** — smaller, focused chunks produce sharper similarity matches

> ¹ Liu, N. et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts.* https://arxiv.org/abs/2307.03172

---

### The Chunking Pipeline

```
DOCUMENT
    │
    ▼
┌───────────────────────────────────────────┐
│           STRATEGY CHOICE                │
│                                           │
│  Fixed-size  │ Recursive │ Sent-Window   │
│              │           │  Semantic     │
└──────────────┬────────────────────────────┘
               │
               ▼  chunks[]
       ┌───────────────┐
       │ Embedding     │  text → vector  [0.12, -0.45 ...]
       │ Model         │
       └───────┬───────┘
               │
               ▼
       ┌───────────────┐
       │  Vector Index │  store + HNSW index
       │  (ChromaDB)   │
       └───────┬───────┘
               │  query time
               ▼
       ┌───────────────┐
       │  Retriever    │  cosine similarity → top-k chunks
       └───────┬───────┘
               │
               ▼
       ┌───────────────┐
       │  LLM          │  generates answer from chunks + query
       └───────────────┘
```

> The chunking strategy determines what goes into the index — everything downstream depends on it.

---

### What Makes a Good Chunk?

| Property | Why it matters |
|----------|----------------|
| **Self-contained** | Can be read in isolation and still make sense |
| **Topic-coherent** | One concept or fact per chunk, not three |
| **Right-sized** | Not so short that context is lost; not so long that retrieval is noisy |
| **Overlap at boundaries** | Key sentences near chunk edges appear in at least one complete chunk |
| **Metadata-rich** | Source, page, section label attached so the LLM can cite provenance |

In [ ]:
# ── Sample document used throughout the workshop ──────────────────────────────
# Five short topic-paragraphs so we can see exactly where each strategy splits.

SAMPLE_TEXT = """Neural networks are computing systems loosely inspired by biological neural networks. Deep learning uses multiple layers to progressively extract higher-level features from raw input. Training these networks requires large labelled datasets and significant compute resources.

The transformer architecture introduced in 2017 revolutionized natural language processing. It replaces recurrent networks with self-attention, allowing all tokens to attend to each other simultaneously. This parallelism enables training on much larger datasets than earlier sequential models.

Retrieval-augmented generation (RAG) combines a language model with an external knowledge base. When a question arrives, a retriever fetches relevant documents, and the generator conditions its output on both the question and the retrieved context. This grounds the model in factual sources and reduces hallucination.

Vector databases store high-dimensional embeddings and support approximate nearest-neighbor search. Systems like ChromaDB, Qdrant, and Pinecone make it practical to search millions of document chunks in milliseconds. The embedding quality is as important as the retrieval algorithm itself.

Evaluation of RAG pipelines requires specialized metrics. Faithfulness checks whether the answer is grounded in the retrieved context. Context recall checks whether all necessary information was retrieved. Answer relevancy checks whether the answer directly addresses the question."""

paragraphs = [p for p in SAMPLE_TEXT.split("\n\n") if p.strip()]
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", SAMPLE_TEXT) if s.strip()]
print(f"Document: {len(SAMPLE_TEXT):,} chars | {len(SAMPLE_TEXT.split()):,} words")
print(f"Paragraphs: {len(paragraphs)}")
print(f"Sentences:  {len(sentences)}")

---

## Part 2 — Fixed-Size Chunking

---

### How It Works

`CharacterTextSplitter` cuts the document every `chunk_size` characters, regardless of where sentences or paragraphs end. An optional `chunk_overlap` copies the last N characters of each chunk into the start of the next.

```
Document: "The transformer architecture introduced in 2017 ..."

chunk_size=100, chunk_overlap=20:

  Chunk 0:  "The transformer architecture introduced in 2017 revolutionized
             natural language proc"
  Chunk 1:  "age proc essing. It replaces recurrent networks with self-attention"
             ─── overlap ───
```

**Tradeoffs:**

| Pro | Con |
|-----|-----|
| Zero config — just set a number | Cuts mid-sentence, mid-word |
| Uniform index size | Context lost at every boundary |
| No API calls needed | Retrieval precision suffers for dense prose |
| Fast — pure character counting | Not aware of headers, lists, or code blocks |

**Best for:** Simple baselines, structured data with no natural sentence boundaries (log files, CSV-as-text), or when uniform shard size is an operational requirement.

In [ ]:
# ─── 2-A: CharacterTextSplitter — basic split ────────────────────────────────

fixed_splitter = CharacterTextSplitter(
    separator="",       # no separator — pure character count
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
fixed_chunks = fixed_splitter.split_text(SAMPLE_TEXT)

print(f"Fixed-size chunks (size=200, overlap=20): {len(fixed_chunks)}\n")
for i, c in enumerate(fixed_chunks):
    print(f"  [{i}] {len(c):3d} chars | {repr(c[:70])}...")

In [ ]:
# ─── 2-B: How chunk_size changes the shard count ─────────────────────────────
# Smaller = more chunks, more precise retrieval, but more index entries.
# Larger  = fewer chunks, more context per chunk, but noisier retrieval.

print(f"{'chunk_size':>10}  {'overlap':>7}  {'chunks':>6}  {'avg chars':>9}")
print("-" * 42)
for size in [100, 200, 300, 500]:
    overlap = size // 10
    splitter = CharacterTextSplitter(separator="", chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_text(SAMPLE_TEXT)
    avg = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{size:>10}  {overlap:>7}  {len(chunks):>6}  {avg:>9.0f}")

---

## Part 3 — Recursive Chunking

---

### How It Works

`RecursiveCharacterTextSplitter` tries a list of separators in priority order. It splits on the largest natural boundary that fits within `chunk_size`, and only falls back to smaller ones when needed:

```
Priority order: ["\n\n", "\n", ". ", " ", ""]
                  │         │      │      │    └─ last resort: character split
                  │         │      │      └────── split on space
                  │         │      └───────────── split on sentence end
                  │         └──────────────────── split on line break
                  └────────────────────────────── split on blank line (paragraph)
```

**Tradeoffs:**

| Pro | Con |
|-----|-----|
| Respects paragraph and sentence boundaries | Still character-based — not truly semantic |
| No API calls needed | Paragraph-heavy docs may produce uneven chunks |
| Configurable separator list | Code blocks or tables may be split mid-structure |
| Default choice for 90% of RAG use cases | |

**Best for:** General prose, blog posts, documentation, legal text, articles. This is the recommended default.

In [ ]:
# ─── 3-A: RecursiveCharacterTextSplitter — default settings ──────────────────

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)

print(f"Recursive chunks (size=300, overlap=30): {len(recursive_chunks)}\n")
for i, c in enumerate(recursive_chunks):
    print(f"  [{i}] {len(c):3d} chars | {repr(c[:70])}...")

In [ ]:
# ─── 3-B: Inspect full chunk contents and boundary quality ───────────────────
# The most important diagnostic: does each chunk read naturally
# as a complete thought? Are the boundaries at sentence ends?

print("=== FULL RECURSIVE CHUNK CONTENTS ===\n")
for i, c in enumerate(recursive_chunks):
    print(f"CHUNK {i}  ({len(c)} chars)")
    print("─" * 60)
    print(c)
    print()

---

## Part 4 — Sentence-Window Chunking

---

### How It Works

Sentence-window chunking separates *what gets embedded* from *what gets stored*:

```
Sentences:   [S0]  [S1]  [S2]  [S3]  [S4]  [S5]

Window=1 around S2:
  Embed  →  "S2"                      ← precise retrieval target
  Store  →  "S1  S2  S3"              ← rich context returned to the LLM
```

The retriever finds the most relevant individual sentence (high precision). But when that chunk is passed to the LLM, it includes its neighbors (high recall of surrounding context).

**Tradeoffs:**

| Pro | Con |
|-----|-----|
| Best precision for fact-retrieval | Index size grows with window size |
| Rich context passed to the LLM | Requires sentence tokenization |
| Works well with parent-document retriever | Window overlap can produce near-duplicate context |

**Best for:** Q&A over dense factual text (legal, medical, technical), when you need high recall of specific claims.

In [ ]:
# ─── 4-A: Sentence-window chunking implementation ────────────────────────────


def sentence_window_chunks(text: str, window: int = 1) -> list:
    """Split text into sentences; each item stores the sentence (embed target)
    and a wider context window (what gets returned to the LLM)."""
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    result = []
    for i, sent in enumerate(sents):
        lo = max(0, i - window)
        hi = min(len(sents), i + window + 1)
        result.append(
            {
                "index": i,
                "sentence": sent,                      # embed this
                "context": " ".join(sents[lo:hi]),     # store this
            }
        )
    return result


sw_chunks = sentence_window_chunks(SAMPLE_TEXT, window=1)

print(f"Sentence-window chunks (window=1): {len(sw_chunks)}")
print()
print("First 3 items — note the embed vs store difference:")
for item in sw_chunks[:3]:
    print(f"  [{item['index']}] EMBED: {repr(item['sentence'][:60])}")
    print(f"       STORE: {repr(item['context'][:80])}")
    print()

In [ ]:
# ─── 4-B: Compare window sizes ───────────────────────────────────────────────
# Window=0: embed and store the sentence alone (maximum precision, minimum context)
# Window=1: +/-1 neighbor (good balance)
# Window=2: +/-2 neighbors (more context, larger stored chunks)

print(f"{'Window':>6}  {'Chunks':>6}  {'Avg embed chars':>15}  {'Avg context chars':>17}")
print("-" * 52)
for w in [0, 1, 2]:
    chunks = sentence_window_chunks(SAMPLE_TEXT, window=w)
    avg_embed = sum(len(c["sentence"]) for c in chunks) / len(chunks)
    avg_ctx = sum(len(c["context"]) for c in chunks) / len(chunks)
    print(f"{w:>6}  {len(chunks):>6}  {avg_embed:>15.0f}  {avg_ctx:>17.0f}")

---

## Part 5 — Semantic Chunking

---

### How It Works

Semantic chunking uses embeddings to detect *topic shifts* between consecutive sentences. When cosine similarity between adjacent sentence embeddings drops below a threshold, the algorithm places a chunk boundary there.

```
Sentences:    S0    S1    S2    S3    S4    S5
              │     │     │     │     │
Similarity:   0.89  0.91  0.61  0.85  0.78
                               ^
              threshold=0.75   │
                         SPLIT HERE (topic changed)

Result:  Chunk A = [S0, S1, S2]  |  Chunk B = [S3, S4, S5]
```

**Tradeoffs:**

| Pro | Con |
|-----|-----|
| Boundaries align to actual topic shifts | Requires an embedding call per sentence |
| Chunks are semantically coherent | Threshold needs tuning per corpus |
| Handles multi-topic documents naturally | Slower and costs API tokens |
| Variable chunk size fits content | May produce very short or very long chunks |

**Best for:** Long documents with distinct sections (research papers, presentations, product specs), where topic coherence matters more than uniform size.

In [ ]:
# ─── 5-A: Semantic chunking implementation ───────────────────────────────────


def cosine_sim(a: List[float], b: List[float]) -> float:
    """Cosine similarity between two embedding vectors."""
    a_arr, b_arr = np.array(a), np.array(b)
    denom = (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)) + 1e-10
    return float(np.dot(a_arr, b_arr) / denom)


def semantic_chunks(
    text: str,
    threshold: float = 0.75,
    emb_model: OpenAIEmbeddings = None,
):
    """Split on cosine similarity drops between consecutive sentence embeddings.

    Returns (chunks, inter_sentence_similarities) where each chunk is a dict
    with keys: chunk_id, text, n_sentences, split_sim.
    """
    if emb_model is None:
        emb_model = OpenAIEmbeddings(model="text-embedding-3-small")

    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    embeddings = emb_model.embed_documents(sents)

    # Cosine similarity between consecutive sentences
    sims = [
        cosine_sim(embeddings[i], embeddings[i + 1])
        for i in range(len(sents) - 1)
    ]

    # Build chunks: split where similarity < threshold
    chunks = []
    current = [sents[0]]

    for i, sim in enumerate(sims):
        if sim < threshold:
            chunks.append(
                {
                    "chunk_id": len(chunks),
                    "text": " ".join(current),
                    "n_sentences": len(current),
                    "split_sim": sim,
                }
            )
            current = [sents[i + 1]]
        else:
            current.append(sents[i + 1])

    if current:
        chunks.append(
            {
                "chunk_id": len(chunks),
                "text": " ".join(current),
                "n_sentences": len(current),
                "split_sim": None,
            }
        )

    return chunks, sims


print("Running semantic chunking (makes embedding API calls)...")
emb_model = OpenAIEmbeddings(model="text-embedding-3-small")
semantic, inter_sims = semantic_chunks(SAMPLE_TEXT, threshold=0.75, emb_model=emb_model)

print(f"Semantic chunks (threshold=0.75): {len(semantic)}\n")
for item in semantic:
    split_info = f"  split_sim={item['split_sim']:.3f}" if item["split_sim"] else "  (last)"
    print(f"  [{item['chunk_id']}] {len(item['text']):3d} chars | {item['n_sentences']} sentences{split_info}")
    print(f"       {repr(item['text'][:70])}...")

In [ ]:
# ─── 5-B: Visualize inter-sentence similarities ──────────────────────────────
# Shows exactly where the topic shifts occur in the document.

THRESHOLD = 0.75

print(f"Inter-sentence cosine similarities (threshold={THRESHOLD})\n")
print(f"{'Pair':>5}  {'Sim':>6}  {'Split?':>6}  Context")
print("-" * 78)
for i, sim in enumerate(inter_sims):
    split_flag = "SPLIT" if sim < THRESHOLD else ""
    marker = "<<<" if sim < THRESHOLD else "   "
    s_a = sentences[i][:30].ljust(30)
    s_b = sentences[i + 1][:25]
    print(f"{i:>5}  {sim:>6.3f}  {split_flag:>6}  {marker} ...{s_a}  →  {s_b}...")

---

## Part 6 — Side-by-Side Comparison

---

### Strategy Comparison Table

| Strategy | Respects structure | Needs embeddings | Chunk size | Best for |
|----------|--------------------|-----------------|------------|----------|
| **Fixed-size** | No | No | Uniform | Baselines, log data, operational uniformity |
| **Recursive** | Yes (paragraph → sentence → word) | No | Near-uniform | General prose — **default choice** |
| **Sentence-window** | Yes (sentence-level) | No | Small embed / large store | Dense factual text, Q&A |
| **Semantic** | Yes (topic-level) | Yes | Variable | Multi-topic documents, research papers |

### Chunk Size vs Overlap Tradeoffs

| Parameter | Low value | High value |
|-----------|-----------|------------|
| `chunk_size` | More chunks, precise retrieval, less context per answer | Fewer chunks, richer context, noisier retrieval |
| `chunk_overlap` | Risk of split facts at boundaries | Higher cost, near-duplicate chunks in index |
| `window` (sentence-window) | Maximum precision, minimum context | Rich context, index bloat |
| `threshold` (semantic) | Fewer, larger chunks (coarse topics) | Many small chunks (fine-grained topics) |

In [ ]:
# ─── 6-A: Numeric summary — same document, all four strategies ───────────────

sw_text_only = [item["sentence"] for item in sw_chunks]   # embed targets
semantic_text_only = [item["text"] for item in semantic]

rows = [
    ("Fixed-size",      fixed_chunks,         "No",  "No"),
    ("Recursive",       recursive_chunks,     "Yes", "No"),
    ("Sentence-window", sw_text_only,         "Yes", "No"),
    ("Semantic",        semantic_text_only,   "Yes", "Yes"),
]

print(f"{'Strategy':<20} {'Chunks':>6} {'Min':>5} {'Avg':>5} {'Max':>5} {'Structure':>10} {'Embeddings':>11}")
print("-" * 68)
for name, chunks, structure, needs_emb in rows:
    sizes = [len(c) for c in chunks]
    mn, avg, mx = min(sizes), sum(sizes) // len(sizes), max(sizes)
    print(f"{name:<20} {len(chunks):>6} {mn:>5} {avg:>5} {mx:>5} {structure:>10} {needs_emb:>11}")

In [ ]:
# ─── 6-B: ASCII size distribution ────────────────────────────────────────────
# A quick visual of how evenly each strategy distributes chunk sizes.


def ascii_histogram(chunks: list, label: str, bar_max: int = 30) -> None:
    sizes = [len(c) for c in chunks]
    if not sizes:
        return
    lo, hi = min(sizes), max(sizes)
    step = max(1, (hi - lo + 1) // 5)
    print(f"{label} ({len(sizes)} chunks)")
    for bucket in range(5):
        blo = lo + bucket * step
        bhi = lo + (bucket + 1) * step
        count = sum(1 for s in sizes if blo <= s < bhi)
        bar = "\u2588" * int(count / max(1, len(sizes)) * bar_max)
        print(f"  {blo:4d}-{bhi:4d} | {bar:<30} {count}")
    print()


ascii_histogram(fixed_chunks,       "Fixed-size")
ascii_histogram(recursive_chunks,   "Recursive")
ascii_histogram(sw_text_only,       "Sentence-window (embed targets)")
ascii_histogram(semantic_text_only, "Semantic")

### Exercise 1 — Tune chunk_size and chunk_overlap

**Goal:** For `RecursiveCharacterTextSplitter` on `SAMPLE_TEXT`, find the `chunk_size` and `chunk_overlap` combination that produces:
- At least 4 chunks
- No chunk shorter than 80 characters
- No chunk longer than 450 characters

**Hint:** Start at `chunk_size=350, chunk_overlap=40` and adjust one parameter at a time.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

# TODO: adjust these values until ALL three conditions are marked PASS
EX1_CHUNK_SIZE = 350
EX1_CHUNK_OVERLAP = 40

ex1_splitter = RecursiveCharacterTextSplitter(
    chunk_size=EX1_CHUNK_SIZE,
    chunk_overlap=EX1_CHUNK_OVERLAP,
)
ex1_chunks = ex1_splitter.split_text(SAMPLE_TEXT)

sizes = [len(c) for c in ex1_chunks]
print(f"Settings: chunk_size={EX1_CHUNK_SIZE}, overlap={EX1_CHUNK_OVERLAP}")
print(f"Chunks: {len(ex1_chunks)}  |  min={min(sizes)}  max={max(sizes)}  avg={sum(sizes)//len(sizes)}")
print()
print(f"  >= 4 chunks:           {'PASS' if len(ex1_chunks) >= 4 else 'FAIL'}")
print(f"  No chunk < 80 chars:   {'PASS' if min(sizes) >= 80 else 'FAIL'}")
print(f"  No chunk > 450 chars:  {'PASS' if max(sizes) <= 450 else 'FAIL'}")

### Exercise 2 — Tune the Semantic Threshold

**Goal:** Run semantic chunking at thresholds `0.60`, `0.75`, and `0.90`. For each, record the chunk count and average chunk size. Which threshold produces chunks most similar in count to the recursive strategy?

**Note:** This makes embedding API calls — roughly 45 sentence-embedding calls total across all three thresholds.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

rec_avg = sum(len(c) for c in recursive_chunks) // len(recursive_chunks)
print(f"Recursive baseline: {len(recursive_chunks)} chunks, avg={rec_avg} chars\n")

for threshold in [0.60, 0.75, 0.90]:
    # TODO: call semantic_chunks(SAMPLE_TEXT, threshold=threshold, emb_model=emb_model)
    # TODO: print threshold, chunk count, and average chunk size
    pass

---

## Part 7 — RAG Quality Check

---

Index each strategy's chunks into a separate Chroma collection, run the same query through all four, and compare the retrieved context and generated answer.

### Evaluation Metrics

| Metric | What it measures | How to check manually |
|--------|-----------------|----------------------|
| **Retrieval precision** | Are the top-k chunks actually relevant? | Read the retrieved chunks |
| **Context coverage** | Does the context contain the answer? | Check if ground truth appears |
| **Answer faithfulness** | Does the LLM answer match the context? | Compare answer to source chunks |
| **Answer relevancy** | Does the answer address the question? | Does it answer what was asked? |
| **Best similarity score** | How close was the top chunk? | Higher = more precise chunking |

In [ ]:
# ─── 7-A: Index all four strategies and run the same question ─────────────────

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
QUESTION = "How does RAG ground model outputs in factual sources?"

strategies = [
    ("Fixed-size",      fixed_chunks),
    ("Recursive",       recursive_chunks),
    ("Sentence-window", sw_text_only),
    ("Semantic",        semantic_text_only),
]

results = {}

for name, chunks in strategies:
    collection_name = f"chunk-{name.lower().replace(' ', '-')}"
    vs = Chroma.from_texts(chunks, emb_model, collection_name=collection_name)

    # Retrieve top-2 chunks
    docs = vs.similarity_search_with_score(QUESTION, k=2)
    context = "\n\n".join(d.page_content for d, _ in docs)
    best_score = 1 - docs[0][1]  # cosine distance → similarity

    # Generate answer
    prompt = (
        f"Answer the question using ONLY the context below. "
        f"If the answer is not in the context, say 'Not found.'\n\n"
        f"Context:\n{context}\n\nQuestion: {QUESTION}"
    )
    answer = llm.invoke(prompt).content

    results[name] = {
        "best_similarity": best_score,
        "context_chars": len(context),
        "context_preview": context[:200],
        "answer": answer,
    }

print(f"Question: '{QUESTION}'\n")
print(f"{'Strategy':<20} {'Sim':>6} {'Context chars':>13} {'Answer (first 80 chars)'}")
print("-" * 80)
for name, r in results.items():
    print(f"{name:<20} {r['best_similarity']:>6.3f} {r['context_chars']:>13} {r['answer'][:80]}...")

In [ ]:
# ─── 7-B: Full answer comparison ─────────────────────────────────────────────

print(f"Question: '{QUESTION}'\n")
print("=" * 60)
for name, r in results.items():
    print(f"\n[{name}]")
    print(f"  Best similarity: {r['best_similarity']:.3f}")
    print(f"  Context ({r['context_chars']} chars):")
    print(f"    {r['context_preview'].replace(chr(10), ' ')[:200]}...")
    print(f"  Answer:")
    print(f"    {r['answer']}")

---

## What's Next?

You now understand the four major chunking strategies and how to measure their impact on retrieval quality. Here's where to go deeper:

### Measure what you built
- **Example 16 — RAG Evaluation with RAGAS** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): score your pipeline on Faithfulness, Answer Relevance, and Context Recall automatically. Open it next — it uses the same ChromaDB + chunking patterns you just learned.

### Improve your chunking
- **Parent Document Retriever**: index small child chunks for retrieval precision, but return the full parent document to the LLM — combines the best of sentence-window and recursive
- **`MarkdownHeaderTextSplitter`**: structure-aware splitting for Markdown — keeps heading hierarchy in metadata
- **`HTMLHeaderTextSplitter`**: splits web pages at `<h1>` / `<h2>` boundaries
- **Late Chunking**: embed the full document first, then pool per-token embeddings per chunk for better long-range context

### Improve retrieval
- **Hybrid search**: combine semantic search with BM25 keyword search for better precision on exact terms
- **Re-ranking**: use a cross-encoder to re-rank chunks after retrieval (`ContextualCompressionRetriever`)
- **MMR (Maximal Marginal Relevance)**: retrieve diverse chunks instead of near-duplicates — `vs.max_marginal_relevance_search()`

### Further reading
- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
- Gao, Y. et al. (2023). *Retrieval-Augmented Generation for LLMs: A Survey.* https://arxiv.org/abs/2312.10997
- LangChain Text Splitters docs: https://python.langchain.com/docs/concepts/text_splitters/

---

*Workshop complete. Next step: Example 16 — put a score on what you just built.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Tune chunk_size and chunk_overlap

**Sample answer:** `chunk_size=350, chunk_overlap=40` already passes all three conditions for this document. If you increase `chunk_size` beyond ~400, the paragraph containing the RAG description will exceed 450 characters. If you decrease `chunk_overlap` below ~20, the paragraph boundaries become clean enough that fewer than 4 chunks may be produced.

**What good output looks like:**
- Each chunk reads as a complete paragraph or meaningful excerpt
- The `min` size stays well above 80 characters (sentence fragments are a warning sign)
- The `max` size stays below 450 characters so retrieval stays precise

**Rule of thumb:** For general prose, `chunk_size = 300–500` with `chunk_overlap = 10% of chunk_size` is a reliable starting point. Adjust based on the average paragraph length in your corpus.

In [ ]:
# Exercise 1 — sample solution
ex1_answer = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=40)
ex1_result = ex1_answer.split_text(SAMPLE_TEXT)
sizes = [len(c) for c in ex1_result]

print(f"chunk_size=350, overlap=40")
print(f"Chunks: {len(ex1_result)}  |  min={min(sizes)}  max={max(sizes)}  avg={sum(sizes)//len(sizes)}")
print(f"  >= 4 chunks:           {'PASS' if len(ex1_result) >= 4 else 'FAIL'}")
print(f"  No chunk < 80 chars:   {'PASS' if min(sizes) >= 80 else 'FAIL'}")
print(f"  No chunk > 450 chars:  {'PASS' if max(sizes) <= 450 else 'FAIL'}")
print()
for i, c in enumerate(ex1_result):
    print(f"[{i}] {len(c)} chars — {repr(c[:80])}...")

### Exercise 2 — Tune the Semantic Threshold

**Sample answer:**

| Threshold | Expected chunks | Expected avg chars | Interpretation |
|-----------|----------------|--------------------|----------------|
| 0.60 | 2–3 | 400–550 | Only splits at very strong topic shifts |
| 0.75 | 3–5 | 220–360 | Balanced — roughly matches recursive on this document |
| 0.90 | 5–9 | 120–200 | Splits at even slight wording changes — may over-fragment |

**Key insight:** The optimal threshold is corpus-dependent. A threshold around `0.70–0.78` tends to match recursive chunking for well-structured prose. For documents with abrupt topic changes (slide decks, FAQs), a lower threshold (0.55–0.65) works better. For flowing narrative essays, a higher threshold (0.80+) avoids over-fragmentation. Always check sample chunks visually before indexing.

In [ ]:
# Exercise 2 — sample solution
rec_avg = sum(len(c) for c in recursive_chunks) // len(recursive_chunks)
print(f"Recursive baseline: {len(recursive_chunks)} chunks, avg={rec_avg} chars\n")

print(f"{'Threshold':>10} {'Chunks':>6} {'Avg chars':>9} {'vs Recursive'}")
print("-" * 44)
for threshold in [0.60, 0.75, 0.90]:
    chunks_ex, _ = semantic_chunks(SAMPLE_TEXT, threshold=threshold, emb_model=emb_model)
    texts = [c["text"] for c in chunks_ex]
    avg = sum(len(t) for t in texts) // len(texts) if texts else 0
    diff = avg - rec_avg
    direction = f"+{diff}" if diff >= 0 else str(diff)
    print(f"{threshold:>10.2f} {len(texts):>6} {avg:>9}   avg {direction} chars vs recursive")